# Mining growth-related items from medical QA datasets

This notebook searches four Hugging Face medical QA datasets for growth-related questions, tags the hits by task type, and exports a CSV for advisor review.

## Datasets targeted
- `GBaker/MedQA-USMLE-4-options`
- `openlifescienceai/medmcqa`
- `qiaojin/PubMedQA`
- `dvilares/head_qa` with a fallback to `bigbio/head_qa`

## What the notebook does
1. Loads each dataset from Hugging Face.
2. Normalizes text fields across different schemas.
3. Searches for growth-related content using a keyword bank.
4. Applies light rule-based tagging for task type.
5. Exports a CSV for advisor review.

You can adjust the keyword bank, task rules, and dataset split settings below.


In [1]:
# If needed, uncomment this cell the first time you run the notebook.
# %pip install -q datasets pandas pyarrow tqdm


In [2]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset, Dataset, DatasetDict


/Users/dominiero/opt/anaconda3/envs/py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

You can change the output directory, the dataset splits to inspect, and whether to limit the number of rows for a quick dry run.


In [3]:
OUTPUT_DIR = Path("growth_qa_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

QUICK_RUN = False      # True = scan fewer rows for debugging
ROW_LIMIT = 5000 if QUICK_RUN else None

DATASET_SPECS = {
    "medqa": {
        "candidates": ["GBaker/MedQA-USMLE-4-options", "GBaker/MedQA-USMLE-4-options-hf"],
        "splits": ["train", "validation", "test"],
    },
    "medmcqa": {
        "candidates": ["openlifescienceai/medmcqa", "araag2/MedMCQA"],
        "splits": ["train", "validation", "test"],
    },
    "pubmedqa": {
        "candidates": ["qiaojin/PubMedQA"],
        "configs": ["pqa_labeled"],
        "splits": ["train"],
    },
    "headqa": {
        "candidates": ["dvilares/head_qa", "bigbio/head_qa"],
        "splits": ["train", "validation", "test"],
    },
    "afrimedqa": {
        "candidates": ["afrimedqa/afrimedqa_v2"],
        "splits": ["train", "validation", "test"],
        "require_specialty_pediatrics": True,
        "require_pediatric_context": True,
    },
    "medical_reasoning": {
        "candidates": ["mamachang/medical-reasoning"],
        "splits": ["train", "validation", "test"],
        "require_pediatric_context": True,
    },
    "differential_diagnosis": {
        "candidates": ["shuyuej/Differential-Diagnosis-Dataset"],
        "splits": ["train", "validation", "test"],
        "require_pediatric_context": True,
    },
    "medxpertqa": {
        "candidates": ["TsinghuaC3I/MedXpertQA"],
        "splits": ["train", "validation", "test"],
        "require_pediatric_context": True,
    },
}


## Keyword bank


In [4]:
KEYWORD_GROUPS = {
    "chart_terms": [
        "growth chart", "growth charts", "percentile", "percentiles", "centile", "centiles",
        "z score", "z-score", "standard deviation score", "sds", "lms", "weight-for-age",
        "length-for-age", "stature-for-age", "bmi-for-age", "weight-for-length",
        "head circumference", "head-circumference", "growth standard", "growth standards",
        "growth reference", "growth velocity", "crossing percentiles", "corrected age",
    ],
    "growth_disorders": [
        "short stature", "tall stature", "growth failure", "failure to thrive",
        "faltering growth", "growth hormone deficiency", "gh deficiency", "igf-1",
        "bone age", "constitutional delay", "familial short stature", "delayed growth",
        "small for gestational age", "sga", "large for gestational age", "lga",
        "intrauterine growth restriction", "iugr", "fetal growth restriction", "fgr",
    ],
    "syndromes_conditions": [
        "turner syndrome", "down syndrome", "noonan syndrome", "achondroplasia",
        "prader-willi", "silver-russell", "russell-silver", "marfan syndrome",
        "precocious puberty", "delayed puberty", "pituitary dwarfism",
    ],
    "pediatric_context": [
        "pediatrician", "well-child", "well child", "infant", "newborn", "neonate",
        "child", "adolescent", "puberty", "tanner stage", "gestational age", "preterm",
        "premature infant", "corrected gestational age", "length", "height", "stature",
    ],
}

ALL_KEYWORDS = sorted({kw for values in KEYWORD_GROUPS.values() for kw in values}, key=len, reverse=True)
KEYWORD_PATTERN = re.compile("|".join(re.escape(k) for k in ALL_KEYWORDS), flags=re.IGNORECASE)


## Task tagging rules


In [5]:
TASK_RULES = {
    "chart_selection": [
        "growth chart", "growth charts", "growth standard", "growth standards",
        "growth reference", "which chart", "percentile", "centile", "corrected age",
        "weight-for-age", "length-for-age", "stature-for-age", "bmi-for-age",
        "weight-for-length", "head circumference",
    ],
    "percentile_or_quantitative_interpretation": [
        "percentile", "percentiles", "centile", "centiles", "z score", "z-score", "sds",
        "lms", "growth velocity", "crossing percentiles", "bone age",
    ],
    "growth_disorder_differential": [
        "short stature", "tall stature", "growth failure", "failure to thrive",
        "growth hormone deficiency", "gh deficiency", "igf-1", "constitutional delay",
        "familial short stature", "delayed puberty", "precocious puberty",
    ],
    "prematurity_or_perinatal_growth": [
        "preterm", "premature infant", "gestational age", "corrected age",
        "small for gestational age", "large for gestational age", "iugr", "fgr",
    ],
    "syndrome_specific_growth": [
        "turner syndrome", "down syndrome", "noonan syndrome", "achondroplasia",
        "prader-willi", "silver-russell", "russell-silver", "marfan syndrome",
    ],
}


In [6]:
def normalize_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, (int, float, bool)):
        return str(value)
    if isinstance(value, list):
        return " | ".join(normalize_text(v) for v in value if v is not None)
    if isinstance(value, dict):
        parts = []
        for k, v in value.items():
            parts.append(f"{k}: {normalize_text(v)}")
        return " | ".join(parts)
    return str(value)

def first_nonempty(row, keys):
    for key in keys:
        if key in row:
            value = normalize_text(row.get(key)).strip()
            if value:
                return value
    return ""

def build_context_from_keys(row, keys):
    parts = []
    for key in keys:
        if key in row:
            value = normalize_text(row.get(key)).strip()
            if value:
                parts.append(f"{key}: {value}")
    return " | ".join(parts)

def join_nonempty(parts):
    return "\n".join(p for p in parts if p and str(p).strip())

def find_matches(text):
    return sorted({m.group(0) for m in KEYWORD_PATTERN.finditer(text)})

def assign_task_types(text):
    lower = text.lower()
    tags = []
    for task, keywords in TASK_RULES.items():
        if any(k.lower() in lower for k in keywords):
            tags.append(task)
    if not tags:
        tags = ["other_growth_adjacent"]
    return tags

def relevance_score(task_types, matches):
    high_value = {"chart_selection", "percentile_or_quantitative_interpretation", "prematurity_or_perinatal_growth", "syndrome_specific_growth"}
    medium_value = {"growth_disorder_differential"}
    score = 1
    if any(t in high_value for t in task_types):
        score = 3
    elif any(t in medium_value for t in task_types):
        score = 2
    return score

def recommendation(score):
    if score == 3:
        return "direct_reuse_or_small_rewrite"
    if score == 2:
        return "rewrite_into_growth_benchmark"
    return "adjacent_background_only"

PEDIATRIC_CONTEXT_PATTERN = re.compile(
    r"\bpediatrics?\b|\bpediatric\b|\bchild(?:ren)?\b|\binfant\b|\bnewborn\b|\bneonate\b|\badolescent\b|\bpuberty\b",
    flags=re.IGNORECASE,
)

HIGH_PRIORITY_CONTEXT_TERMS = [
    "assessment of growth",
    "growth",
    "growth and development",
    "growth, development, and behavior",
    "sho stature",
    "short stature",
    "endocrinology",
    "disorders of pituitary gland",
    "disorders of pubey",
    "disorders of puberty",
    "pubey and adolescent health",
    "puberty and adolescent health",
    "adolescence",
]
HIGH_PRIORITY_THEME_PATTERN = re.compile("|".join(re.escape(t) for t in HIGH_PRIORITY_CONTEXT_TERMS), flags=re.IGNORECASE)

def has_pediatric_context(text):
    return bool(PEDIATRIC_CONTEXT_PATTERN.search(normalize_text(text)))

def specialty_mentions_pediatrics(row):
    specialty_fields = [
        "specialty", "speciality", "specialties", "specialities",
        "subject", "subject_name", "topic", "topic_name", "category",
    ]
    text = " | ".join(normalize_text(row.get(k)) for k in specialty_fields if k in row)
    return bool(re.search(r"\bpediatrics?\b|\bpediatric\b", text, flags=re.IGNORECASE))


## Dataset adapters


In [7]:
def adapt_medqa(row):
    options_text = normalize_text(row.get("options"))
    answer_text = normalize_text(row.get("answer"))
    question_text = normalize_text(row.get("question"))
    context_text = normalize_text(row.get("context"))
    explanation_text = normalize_text(row.get("explanation"))
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

def adapt_medmcqa(row):
    options = [row.get("opa"), row.get("opb"), row.get("opc"), row.get("opd")]
    options_text = " | ".join([f"{label}: {normalize_text(opt)}" for label, opt in zip(list("ABCD"), options)])
    answer_text = normalize_text(row.get("cop"))
    explanation_text = normalize_text(row.get("exp"))
    question_text = normalize_text(row.get("question"))
    context_text = " | ".join([
        f"subject: {normalize_text(row.get('subject_name'))}",
        f"topic: {normalize_text(row.get('topic_name'))}",
    ])
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

def adapt_pubmedqa(row):
    question_text = normalize_text(row.get("question"))
    context_text = normalize_text(row.get("context"))
    answer_text = normalize_text(row.get("final_decision"))
    explanation_text = normalize_text(row.get("long_answer"))
    options_text = "yes | no | maybe"
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

def adapt_headqa(row):
    question_text = normalize_text(row.get("qtext"))
    context_text = normalize_text(row.get("category"))
    options_text = normalize_text(row.get("answers") or row.get("atext"))
    answer_text = normalize_text(row.get("ra"))
    explanation_text = normalize_text(row.get("image"))
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

def adapt_afrimedqa(row):
    question_text = first_nonempty(row, ["question", "query", "stem", "prompt"])
    context_text = build_context_from_keys(row, ["specialty", "subject", "topic", "category", "context"])
    options_text = first_nonempty(row, ["options", "choices", "mcq_options", "candidates"])
    answer_text = first_nonempty(row, ["answer", "final_answer", "correct_answer", "label"])
    explanation_text = first_nonempty(row, ["explanation", "rationale", "reasoning"])
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

def adapt_medical_reasoning(row):
    question_text = first_nonempty(row, ["question", "query", "problem", "case", "stem"])
    context_text = build_context_from_keys(row, ["specialty", "subject", "topic", "category", "context", "clinical_context"])
    options_text = first_nonempty(row, ["options", "choices", "candidates"])
    answer_text = first_nonempty(row, ["answer", "final_answer", "gold", "label"])
    explanation_text = first_nonempty(row, ["reasoning", "rationale", "explanation", "analysis"])
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

def adapt_differential_diagnosis(row):
    question_text = first_nonempty(row, ["question", "query", "chief_complaint", "case", "stem"])
    context_text = build_context_from_keys(row, ["specialty", "subject", "topic", "category", "context"])
    options_text = first_nonempty(row, ["differentials", "options", "choices", "candidate_diagnoses"])
    answer_text = first_nonempty(row, ["answer", "final_diagnosis", "gold", "label"])
    explanation_text = first_nonempty(row, ["rationale", "reasoning", "explanation"])
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

def adapt_medxpertqa(row):
    question_text = first_nonempty(row, ["question", "query", "prompt", "stem"])
    context_text = build_context_from_keys(row, ["specialty", "subject", "topic", "discipline", "context"])
    options_text = first_nonempty(row, ["options", "choices", "candidates", "answers"])
    answer_text = first_nonempty(row, ["answer", "final_answer", "correct_answer", "label"])
    explanation_text = first_nonempty(row, ["reasoning", "rationale", "explanation", "analysis"])
    full_text = join_nonempty([question_text, context_text, options_text, answer_text, explanation_text])
    return {
        "question_text": question_text,
        "context_text": context_text,
        "options_text": options_text,
        "answer_text": answer_text,
        "explanation_text": explanation_text,
        "full_text": full_text,
    }

ADAPTERS = {
    "medqa": adapt_medqa,
    "medmcqa": adapt_medmcqa,
    "pubmedqa": adapt_pubmedqa,
    "headqa": adapt_headqa,
    "afrimedqa": adapt_afrimedqa,
    "medical_reasoning": adapt_medical_reasoning,
    "differential_diagnosis": adapt_differential_diagnosis,
    "medxpertqa": adapt_medxpertqa,
}


In [8]:
def load_first_working_dataset(name, spec):
    errors = []
    candidates = spec.get("candidates", [])
    configs = spec.get("configs", [None])

    for ds_name in candidates:
        for config in configs:
            try:
                if config is None:
                    ds = load_dataset(ds_name)
                else:
                    ds = load_dataset(ds_name, config)
                print(f"Loaded {name}: {ds_name}" + (f" / config={config}" if config else ""))
                return ds_name, config, ds
            except Exception as e:
                errors.append(f"{ds_name}" + (f" / config={config}" if config else "") + f" -> {type(e).__name__}: {e}")
    raise RuntimeError(f"Could not load any candidate for {name}.\n" + "\n".join(errors))

def get_available_splits(ds_obj):
    if isinstance(ds_obj, DatasetDict):
        return list(ds_obj.keys())
    if isinstance(ds_obj, Dataset):
        return ["train"]
    return []

def get_split(ds_obj, split_name):
    if isinstance(ds_obj, DatasetDict):
        return ds_obj[split_name]
    if isinstance(ds_obj, Dataset) and split_name == "train":
        return ds_obj
    raise KeyError(split_name)


## Scan datasets and collect growth-related hits


In [9]:
records = []
dataset_load_log = []

for dataset_key, spec in DATASET_SPECS.items():
    try:
        ds_name, config_name, ds_obj = load_first_working_dataset(dataset_key, spec)
        dataset_load_log.append({"dataset_key": dataset_key, "dataset_name": ds_name, "config": config_name, "status": "loaded"})
    except Exception as e:
        dataset_load_log.append({"dataset_key": dataset_key, "dataset_name": None, "config": None, "status": f"FAILED: {e}"})
        continue

    available_splits = get_available_splits(ds_obj)
    wanted_splits = [s for s in spec["splits"] if s in available_splits]
    adapter = ADAPTERS[dataset_key]

    for split in wanted_splits:
        ds_split = get_split(ds_obj, split)
        n_rows = len(ds_split)
        if ROW_LIMIT is not None:
            n_rows = min(n_rows, ROW_LIMIT)

        print(f"Scanning {dataset_key} / {split} / rows={n_rows}")
        iterator = ds_split.select(range(n_rows)) if ROW_LIMIT is not None else ds_split

        for idx, row in enumerate(tqdm(iterator, leave=False)):
            adapted = adapter(row)

            if spec.get("require_specialty_pediatrics") and not specialty_mentions_pediatrics(row):
                continue

            if spec.get("require_pediatric_context") and not has_pediatric_context(adapted.get("full_text", "")):
                continue

            text = adapted["full_text"]
            matches = find_matches(text)
            if not matches:
                continue

            task_types = assign_task_types(text)
            score = relevance_score(task_types, matches)

            records.append({
                "dataset_key": dataset_key,
                "dataset_name": ds_name,
                "config_name": config_name,
                "split": split,
                "row_index": idx,
                "matched_terms": "; ".join(matches),
                "n_matches": len(matches),
                "task_types": "; ".join(task_types),
                "relevance_score": score,
                "recommendation": recommendation(score),
                "question_text": adapted["question_text"],
                "context_text": adapted["context_text"],
                "options_text": adapted["options_text"],
                "answer_text": adapted["answer_text"],
                "explanation_text": adapted["explanation_text"],
                "full_text": adapted["full_text"][:12000],
            })

load_log_df = pd.DataFrame(dataset_load_log)
hits_df = pd.DataFrame(records)

print("Loaded datasets:")
display(load_log_df)

print(f"Total candidate hits: {len(hits_df):,}")
display(hits_df.head(10))


Loaded medqa: GBaker/MedQA-USMLE-4-options
Scanning medqa / train / rows=10178


Scanning medqa / test / rows=1273


Loaded medmcqa: openlifescienceai/medmcqa
Scanning medmcqa / train / rows=182822


Scanning medmcqa / validation / rows=4183


Scanning medmcqa / test / rows=6150


...

Loaded differential_diagnosis: shuyuej/Differential-Diagnosis-Dataset
Scanning differential_diagnosis / train / rows=1871


Loaded datasets:


,dataset_key,dataset_name,config,status
0,medqa,GBaker/MedQA-USMLE-4-options,None,loaded
1,medmcqa,openlifescienceai/medmcqa,None,loaded
2,pubmedqa,qiaojin/PubMedQA,pqa_labeled,loaded
3,headqa,None,None,FAILED: Could not load any candidate for headq...
4,afrimedqa,afrimedqa/afrimedqa_v2,None,loaded
5,medical_reasoning,mamachang/medical-reasoning,None,loaded
6,differential_diagnosis,shuyuej/Differential-Diagnosis-Dataset,None,loaded
7,medxpertqa,None,None,FAILED: Could not load any candidate for medxp...


Total candidate hits: 27,042


,dataset_key,dataset_name,config_name,split,row_index,matched_terms,n_matches,task_types,relevance_score,recommendation,question_text,context_text,options_text,answer_text,explanation_text,full_text
0,medqa,GBaker/MedQA-USMLE-4-options,None,train,1,infant,1,other_growth_adjacent,1,adjacent_background_only,A 3-month-old baby died suddenly at night whil...,,A: Placing the infant in a supine position on ...,Placing the infant in a supine position on a f...,,A 3-month-old baby died suddenly at night whil...
1,medqa,GBaker/MedQA-USMLE-4-options,None,train,2,child; infant; pediatrician,3,other_growth_adjacent,1,adjacent_background_only,A mother brings her 3-week-old infant to the p...,,A: Abnormal migration of ventral pancreatic bu...,Abnormal migration of ventral pancreatic bud,,A mother brings her 3-week-old infant to the p...
2,medqa,GBaker/MedQA-USMLE-4-options,None,train,7,infant,1,other_growth_adjacent,1,adjacent_background_only,A 3900-g (8.6-lb) male infant is delivered at ...,,A: Gastric fundus in the thorax | B: Pancreati...,Gastric fundus in the thorax,,A 3900-g (8.6-lb) male infant is delivered at ...
3,medqa,GBaker/MedQA-USMLE-4-options,None,train,10,lms,1,percentile_or_quantitative_interpretation,3,direct_reuse_or_small_rewrite,A 46-year-old woman comes to the physician bec...,,A: Granulomatous inflammation of the cavernous...,Glycosaminoglycan accumulation in the orbit,,A 46-year-old woman comes to the physician bec...
4,medqa,GBaker/MedQA-USMLE-4-options,None,train,26,Tanner stage; precocious puberty,2,growth_disorder_differential,2,rewrite_into_growth_benchmark,A 5-year-old girl is brought to the clinic by ...,,A: Granulosa cell tumor | B: Idiopathic precoc...,Granulosa cell tumor,,A 5-year-old girl is brought to the clinic by ...
5,medqa,GBaker/MedQA-USMLE-4-options,None,train,41,neonate; pediatrician,2,other_growth_adjacent,1,adjacent_background_only,A male neonate is being examined by a pediatri...,,A: Atrial septal defect | B: Ventricular septa...,Patent ductus arteriosus,,A male neonate is being examined by a pediatri...
6,medqa,GBaker/MedQA-USMLE-4-options,None,train,42,child,1,other_growth_adjacent,1,adjacent_background_only,A 4-year-old boy is brought to the emergency d...,,A: Production of IL-2 by Th1 cells | B: Activa...,Formation of C5-9 complex,,A 4-year-old boy is brought to the emergency d...
7,medqa,GBaker/MedQA-USMLE-4-options,None,train,55,infant; newborn,2,other_growth_adjacent,1,adjacent_background_only,You are examining a 3-day-old newborn who was ...,,A: Phenylalanine hydroxylase | B: Branched-cha...,Carbamoyl phosphate synthetase I,,You are examining a 3-day-old newborn who was ...
8,medqa,GBaker/MedQA-USMLE-4-options,None,train,68,infant; length; well-child,3,other_growth_adjacent,1,adjacent_background_only,A 5-week-old infant born at 36 weeks' gestatio...,,A: Prostaglandin E1 infusion | B: Indomethacin...,Indomethacin infusion,,A 5-week-old infant born at 36 weeks' gestatio...
9,medqa,GBaker/MedQA-USMLE-4-options,None,train,73,child,1,other_growth_adjacent,1,adjacent_background_only,"A 31-year-old woman, gravida 2, para 1, at 32 ...",,"A: Administer betamethasone, ampicillin, and p...",Administer betamethasone and ampicillin,,"A 31-year-old woman, gravida 2, para 1, at 32 ..."


## Add review-oriented columns


In [10]:
if not hits_df.empty:
    hits_df["pediatric_flag"] = hits_df["full_text"].str.contains(r"\binfant\b|\bnewborn\b|\bchild\b|\badolescent\b|\bpediatric", case=False, regex=True)
    hits_df["chart_related_flag"] = hits_df["full_text"].str.contains(r"growth chart|percentile|centile|weight-for-age|length-for-age|stature-for-age|bmi-for-age|head circumference|corrected age", case=False, regex=True)
    hits_df["syndrome_specific_flag"] = hits_df["full_text"].str.contains(r"turner syndrome|down syndrome|noonan syndrome|achondroplasia|prader-willi|silver-russell|russell-silver", case=False, regex=True)
    hits_df["prematurity_flag"] = hits_df["full_text"].str.contains(r"preterm|premature infant|gestational age|corrected age|small for gestational age|large for gestational age|iugr|fgr", case=False, regex=True)
    hits_df["quantitative_flag"] = hits_df["full_text"].str.contains(r"percentile|centile|z score|z-score|sds|lms|growth velocity|bone age", case=False, regex=True)

    review_columns = [
        "dataset_key", "dataset_name", "config_name", "split", "row_index",
        "relevance_score", "recommendation", "task_types", "matched_terms",
        "pediatric_flag", "chart_related_flag", "syndrome_specific_flag",
        "prematurity_flag", "quantitative_flag",
        "question_text", "answer_text", "context_text", "options_text", "explanation_text",
    ]

    review_df = hits_df[review_columns].copy()
else:
    review_df = pd.DataFrame()


## Summary tables


In [11]:
if not review_df.empty:
    by_dataset = review_df.groupby("dataset_key").size().rename("n_hits").reset_index().sort_values("n_hits", ascending=False)
    by_task = (
        review_df.assign(task_type=review_df["task_types"].str.split("; "))
        .explode("task_type")
        .groupby("task_type").size().rename("n_hits").reset_index()
        .sort_values("n_hits", ascending=False)
    )
    by_recommendation = review_df.groupby("recommendation").size().rename("n_hits").reset_index().sort_values("n_hits", ascending=False)

    display(by_dataset)
    display(by_task)
    display(by_recommendation)
else:
    print("No hits found. Expand the keyword bank or check dataset loading.")


,dataset_key,n_hits
1,medmcqa,24342
2,medqa,2045
3,pubmedqa,351
0,afrimedqa,304


,task_type,n_hits
2,other_growth_adjacent,22080
3,percentile_or_quantitative_interpretation,1689
4,prematurity_or_perinatal_growth,1459
5,syndrome_specific_growth,1188
1,growth_disorder_differential,770
0,chart_selection,701


,recommendation,n_hits
0,adjacent_background_only,22080
1,direct_reuse_or_small_rewrite,4359
2,rewrite_into_growth_benchmark,603


## Export CSVs


In [12]:
load_log_df.to_csv(OUTPUT_DIR / "dataset_load_log.csv", index=False)

if not hits_df.empty:
    hits_df.to_csv(OUTPUT_DIR / "growth_related_candidates_full.csv", index=False)
    review_df.to_csv(OUTPUT_DIR / "growth_related_candidates_review.csv", index=False)

    context_series = review_df["context_text"].fillna("").astype(str)
    strict_context_filter = (
        context_series.str.contains(r"\bpediatrics?\b|\bpediatric\b", case=False, regex=True)
        & context_series.str.contains(HIGH_PRIORITY_THEME_PATTERN)
    )

    high_priority_df = review_df[(review_df["relevance_score"] >= 2) & strict_context_filter].sort_values(
        ["relevance_score", "dataset_key"], ascending=[False, True]
    )
    high_priority_df.to_csv(OUTPUT_DIR / "growth_related_high_priority.csv", index=False)

    print("Saved:")
    for p in sorted(OUTPUT_DIR.glob("*.csv")):
        print(" -", p)
else:
    print("No candidate CSVs were written because no hits were found.")


Saved:
 - growth_qa_outputs/dataset_load_log.csv
 - growth_qa_outputs/growth_related_candidates_full.csv
 - growth_qa_outputs/growth_related_candidates_review.csv
 - growth_qa_outputs/growth_related_high_priority.csv


## Optional: semantic reranking with seed queries

Keyword search is still the high-recall first pass, but you can add embeddings to rank the matched rows by semantic closeness to a small set of target growth prompts.

This is useful when the row is relevant but does not literally repeat phrases like `growth chart` or `short stature differential diagnosis`.

The cell below:
1. Embeds each row in `hits_df["full_text"]`
2. Embeds a small list of seed queries
3. Assigns each row its best-matching seed query and cosine similarity score
4. Exports a semantically reranked CSV for manual review

If `sentence-transformers` is not installed yet, run:
`%pip install -q sentence-transformers`


In [13]:
SEED_QUERIES = [
    "pediatric growth assessment",
    "short stature differential diagnosis",
    "which growth chart should be used",
    "premature infant corrected age growth",
]

SEMANTIC_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_TOP_N = 250

def semantic_rerank_hits(candidates_df, seed_queries, text_col="full_text", model_name=SEMANTIC_MODEL_NAME):
    from sentence_transformers import SentenceTransformer
    
    texts = candidates_df[text_col].fillna("").astype(str).tolist()
    model = SentenceTransformer(model_name)

    doc_embeddings = model.encode(
        texts,
        batch_size=128,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    query_embeddings = model.encode(seed_queries, normalize_embeddings=True)

    similarity = query_embeddings @ doc_embeddings.T
    best_query_idx = similarity.argmax(axis=0)
    best_scores = similarity.max(axis=0)

    reranked = candidates_df.copy()
    reranked["semantic_seed_query"] = [seed_queries[i] for i in best_query_idx]
    reranked["semantic_score"] = best_scores
    reranked = reranked.sort_values(
        ["semantic_score", "relevance_score", "n_matches"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    return reranked

if hits_df.empty:
    semantic_hits_df = pd.DataFrame()
    semantic_review_df = pd.DataFrame()
    print("No keyword hits available for semantic reranking.")
else:
    semantic_hits_df = semantic_rerank_hits(hits_df, SEED_QUERIES)
    semantic_review_columns = [
        "dataset_key", "dataset_name", "config_name", "split", "row_index",
        "semantic_seed_query", "semantic_score",
        "relevance_score", "recommendation", "task_types", "matched_terms",
        "question_text", "answer_text", "context_text", "options_text", "explanation_text",
    ]
    semantic_review_df = semantic_hits_df[semantic_review_columns].head(SEMANTIC_TOP_N).copy()
    display(semantic_review_df.head(30))

    semantic_hits_df.to_csv(OUTPUT_DIR / "growth_related_candidates_semantic_full.csv", index=False)
    semantic_review_df.to_csv(OUTPUT_DIR / "growth_related_candidates_semantic_top.csv", index=False)
    print("Saved semantic outputs:")
    print(" -", OUTPUT_DIR / "growth_related_candidates_semantic_full.csv")
    print(" -", OUTPUT_DIR / "growth_related_candidates_semantic_top.csv")


Batches: 100%|██████████| 212/212 [02:17<00:00,  1.55it/s]


,dataset_key,dataset_name,config_name,split,row_index,semantic_seed_query,semantic_score,relevance_score,recommendation,task_types,matched_terms,question_text,answer_text,context_text,options_text,explanation_text
0,afrimedqa,afrimedqa/afrimedqa_v2,None,train,14544,short stature differential diagnosis,0.744068,2,rewrite_into_growth_benchmark,growth_disorder_differential,tall stature,The following genetic abnormalities are associ...,"option3,option4",specialty: Pediatrics,,
1,medmcqa,openlifescienceai/medmcqa,None,train,46119,short stature differential diagnosis,0.728566,3,direct_reuse_or_small_rewrite,chart_selection; percentile_or_quantitative_in...,Short stature; centile; centileS; height; shor...,Short stature is defined as a height below:,0,"subject: Pediatrics | topic: Growth, Developme...",A: 3rd centile | B: 5th centile | C: 15th cent...,Ans. A. 3rd centileShort stature is defined as...
2,medmcqa,openlifescienceai/medmcqa,None,train,84132,pediatric growth assessment,0.719722,3,direct_reuse_or_small_rewrite,chart_selection,Head circumference; child; height,Best indicator of growth monitoring in children,2,subject: Pediatrics | topic: Growth,A: Weight | B: Mid-arm circumference | C: Rate...,Rate of increase in height or weight as age ad...
3,afrimedqa,afrimedqa/afrimedqa_v2,None,train,14647,pediatric growth assessment,0.715721,1,adjacent_background_only,other_growth_adjacent,child,Which of the following growth parameters is no...,option1,specialty: Pediatrics,,
4,medqa,GBaker/MedQA-USMLE-4-options,None,train,2052,short stature differential diagnosis,0.709341,3,direct_reuse_or_small_rewrite,chart_selection; percentile_or_quantitative_in...,Familial short stature; Growth hormone deficie...,A 15-year-old male adolescent presents to the ...,Constitutional growth delay,,A: Constitutional growth delay | B: Familial s...,
5,medmcqa,openlifescienceai/medmcqa,None,train,95864,pediatric growth assessment,0.700122,3,direct_reuse_or_small_rewrite,chart_selection,Head circumference; child; height,Best indicator of growth monitoring in childre...,2,subject: Pediatrics | topic:,A: Weight | B: Mid-arm circumference | C: Rate...,Rate of increase in height & weight Midarm cir...
6,medmcqa,openlifescienceai/medmcqa,None,train,147221,short stature differential diagnosis,0.693648,3,direct_reuse_or_small_rewrite,chart_selection; percentile_or_quantitative_in...,Child; Down syndrome; Growth hormone deficienc...,The most common etiology of short stature is,2,subject: Pediatrics | topic:,A: Thyroxine deficiency | B: Growth hormone de...,Short Stature\nDefined as height below the thi...
7,medmcqa,openlifescienceai/medmcqa,None,train,25528,short stature differential diagnosis,0.692538,3,direct_reuse_or_small_rewrite,chart_selection; percentile_or_quantitative_in...,Achondroplasia; Constitutional delay; IUGR; Sh...,Short stature is seen in –a) Maternal deprivat...,3,subject: Pediatrics | topic:,A: abc | B: bcd | C: cde | D: abe,If the height of the child is below the 3rd pe...
8,medmcqa,openlifescienceai/medmcqa,None,train,147310,short stature differential diagnosis,0.691739,2,rewrite_into_growth_benchmark,growth_disorder_differential,Height; Short stature; growth hormone deficiency,"Short stature, secondary to growth hormone def...",0,subject: Pediatrics | topic:,A: Normal body proportion | B: Low birth weigh...,
9,medmcqa,openlifescienceai/medmcqa,None,train,91424,pediatric growth assessment,0.689205,3,direct_reuse_or_small_rewrite,chart_selection,Head circumference; child; growth chart; growt...,Best indicator of growth monitoring in children -,2,"subject: Pediatrics | topic: Nutrition, Food S...",A: Weight | B: Mid-arm circumference | C: Rate...,Ans. is 'c' i.e. Rate of increase in height & ...


Saved semantic outputs:
 - growth_qa_outputs/growth_related_candidates_semantic_full.csv
 - growth_qa_outputs/growth_related_candidates_semantic_top.csv


## Optional: manual review instructions

When reviewing the CSV, add columns like:
- `keep_for_benchmark` (yes / no)
- `rewrite_needed` (none / light / substantial)
- `ideal_new_task_type`
- `notes_for_conversion`
